# Практика · Дистрибутивна семантика

Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
Домашнє завдання: [homework.html](homework.html)

Тут ми **своїми руками** робимо все, що лекція називає числом, і в тому самому
порядку. Жодної бібліотеки для ембедингів: матрицю співвіднесень, PPMI і SVD
пишемо самі, бо саме формула — предмет теми.

Що зробимо:

1. Завантажимо корпус українських перекладів і поріжемо його канонічним токенізатором.
2. Заміряємо **стіну**: косинус TF-IDF на 209 парах документів однакового змісту.
3. Побудуємо матрицю «слово × контекст» — спершу на шести рядках руками, потім на всьому корпусі.
4. Покажемо, чому **сира частота не годиться**, і полагодимо це формулою PPMI.
5. Перевіримо **гіпотезу Гарріса** числом: 38 пар синонімів проти контролю тієї самої частоти.
6. Покрутимо **вікно** й побачимо, що воно міняє вид схожості, а не якість.
7. Стиснемо матрицю через **SVD** і заміряємо, чи пробито стіну.
8. Знайдемо **межу підходу**: антоніми, які виявляються схожішими за синоніми.

> ⏱ Зошит будує матрицю 6074 × 6074 і робить понад тридцять сингулярних розкладів.
> Заміряно: **75 секунд процесорного часу**, а від початку до кінця — **дві-три
> хвилини** на чотирьох ядрах без відеокарти, залежно від того, чим ще зайнята
> машина. Мережа не потрібна жодного разу.

## 1 · Підготовка: фіксуємо потоки до імпорту numpy

Перший рядок зошита — не імпорт, а чотири присвоєння змінних середовища. Це не
забобон: без них numpy запускає стільки потоків OpenMP, скільки на машині ядер, а
ті потоки **крутяться в очікуванні**, і це очікування рахується як процесорний час.
На спільній машині курсу той самий розрахунок «коштував» у 45 разів більше, ніж
насправді.

Змінні мусять стояти **до** `import numpy`, бо бібліотека читає їх один раз при
завантаженні. Далі час міряємо `time.process_time()` — процесорним годинником, а не
стінним: стінний на зайнятій машині показує погоду, а не роботу.

In [ ]:
import os

# фіксуємо потоки ДО імпорту numpy — інакше процесорний час буде завищений у рази
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import collections
import gettext
import glob
import re
import sys
import time

import numpy as np
import scipy
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.utils.extmath import randomized_svd

SEEDS = (0, 1, 2)          # три зерна на кожну точку — вимога курсу
started_at = time.process_time()

print('Python     ', sys.version.split()[0])
print('numpy      ', np.__version__)
print('scipy      ', scipy.__version__)
import sklearn
print('scikit-learn', sklearn.__version__)
print('потоки OMP  ', os.environ['OMP_NUM_THREADS'])

## 2 · Корпус: 93 392 українські переклади інтерфейсів

Той самий корпус, що в темах 01-10. У кожній програмі Linux є файл `.mo` з парами
«англійський оригінал → український переклад». Ми беремо **переклади** як документи.

Чому саме він: це справжня українська мова, вона лежить на диску (жодної мережі) і
не міняється між запусками. Чому він **не** універсальний: це вузький домен —
технічна лексика, короткі речення, багато наказового способу.

Якщо української локалі на машині немає, вмикається вбудований запасний корпус.
Він маленький, зошит із ним виконається до кінця, але числа будуть інші — і зошит
про це чесно скаже.

In [ ]:
def load_system_corpus():
    """Трійки (програма, англійський оригінал, український переклад) з усіх .mo-файлів."""
    found = []
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        try:
            with open(path, 'rb') as handle:
                catalog = gettext.GNUTranslations(handle)
        except Exception:
            continue                      # зламаний або чужий формат — просто пропускаємо
        program = path.split('/')[-1][:-3]
        for source, target in catalog._catalog.items():
            # службовий заголовок каталогу має ключ '' і текстом не є
            if isinstance(source, str) and isinstance(target, str) \
               and len(target) > 30 and 'Project-Id' not in target:
                found.append((program, source, target))
    return found

FALLBACK_TEXT = [
    'не вдалося відкрити файл налаштувань програми',
    'не вдалося відкрити документ налаштувань програми',
    'файл не знайдено на вказаному шляху',
    'документ не знайдено на вказаному шляху',
    'відкрити файл лише для читання без запису',
    'відкрити документ лише для читання без запису',
    'сталася внутрішня помилка під час читання файла',
    'сталася критична помилка під час читання документа',
    'діалогове вікно закрито користувачем без збереження',
    'діалогове вікно згорнуто користувачем без збереження',
    'перевірте налаштування мережі та повторіть спробу',
    'перевірте параметри мережі та повторіть спробу',
]

corpus = load_system_corpus()
using_fallback = len(corpus) < 1000
if using_fallback:
    corpus = [('fallback', text, text) for text in FALLBACK_TEXT * 40]
    print('⚠ української локалі немає — працюємо на запасному корпусі, числа будуть інші')
else:
    print('джерело: /usr/share/locale/uk/LC_MESSAGES/*.mo')

documents = [target for program, source, target in corpus]
print(f'документів: {len(documents)}')
print(f'програм:    {len({program for program, source, target in corpus})}')
print('приклад:   ', documents[0][:70])

## 3 · Канонічний токенізатор блоку

Регулярний вираз узятий дослівно з теми 02 і не міняється в жодній темі курсу.
Він трактує апостроф як **звʼязку всередині слова**: `зʼєднання` — один токен, а не
два. Якби кожна тема взяла свій вираз, числа тем перестали б сходитися між собою —
у блоці 1 розбіжність на 0.5 % мало не розсипала весь блок.

In [ ]:
TOKEN_PATTERN = r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*"     # після lowercase=True
token_re = re.compile(TOKEN_PATTERN)

def tokenize(text):
    """Слова документа малими літерами; апостроф лишається всередині слова."""
    return token_re.findall(text.lower())

tokenized = [tokenize(text) for text in documents]
occurrences = sum(len(words) for words in tokenized)
form_counts = collections.Counter()
for words in tokenized:
    form_counts.update(words)

print(f'слововживань: {occurrences}')
print(f'словоформ:    {len(form_counts)}')
print('перевірка апострофа:', tokenize("не вдалося встановити з'єднання"))

### Той самий корпус, інший токенізатор — інші числа

До появи канону в курсі жив простіший вираз, у якому апостроф ʼ — звичайна літера
всередині класу, а `'` і `’` слово розривають. На тому самому корпусі він дає інші
числа, і різниця не випадкова: наш вираз **зшиває** слово через будь-який із трьох
апострофів, а той — лише через один.

Число саме по собі нічого не означає без правила, яким його дістали. Тому покажімо
обидва поруч, а не будемо вгадувати, звідки взялося розходження.

In [ ]:
OLD_PATTERN = r"[абвгґдежзиійклмнопрстуфхцчшщьюяєіїʼ]+"     # вираз до появи канону
old_re = re.compile(OLD_PATTERN)

old_counts = collections.Counter()
old_total = 0
for text in documents:
    words = old_re.findall(text.lower())
    old_total += len(words)
    old_counts.update(words)

print(f'{"токенізатор":<22}{"слововживань":>14}{"словоформ":>12}')
print(f'{"старий вираз":<22}{old_total:>14}{len(old_counts):>12}')
print(f'{"канон блоку (наш)":<22}{occurrences:>14}{len(form_counts):>12}')
print()
print(f'різниця: {old_total - occurrences} слововживань — це слова з апострофом',
      "'" + ' або ’, які старий вираз розриває навпіл, а ми лишаємо цілими')
print('приклад:', old_re.findall("зʼєднання з'єднання"), 'проти',
      tokenize("зʼєднання з'єднання"))

## 4 · Словник: `min_count = 10`

Слово, яке трапилось у корпусі тричі, нічого не скаже нам про свої контексти — у
нього їх просто немає. Тому все, що трапилось менш ніж **десять** разів, ми
викидаємо. Так робить і Word2Vec, і саме тому далі числа можна буде порівнювати.

Ключова подробиця: рідкісні слова ми викидаємо **до** того, як накладати вікно.
Наприкінці розділу 5 побачимо, скільки коштує зробити навпаки.

In [ ]:
MIN_COUNT = 10
vocabulary = sorted(word for word, times in form_counts.items() if times >= MIN_COUNT)
word_index = {word: i for i, word in enumerate(vocabulary)}
covered = sum(form_counts[word] for word in vocabulary)

kept = [[word for word in words if word in word_index] for words in tokenized]
kept_tokens = sum(len(words) for words in kept)

raw_lengths = np.array([len(words) for words in tokenized])
kept_lengths = np.array([len(words) for words in kept])

print(f'словник при min_count={MIN_COUNT}: {len(vocabulary)} слів')
print(f'покриття слововживань:            {100 * covered / occurrences:.4f} %')
print(f'токенів після викидання:          {kept_tokens}')
print(f'медіана довжини документа:        {np.median(raw_lengths):.0f} слів, '
      f'після викидання {np.median(kept_lengths):.0f}')

## 5 · Стіна, з якої починається тема

Нам пощастило з корпусом: один англійський рядок різні перекладачі переклали
по-різному, і обидва переклади лежать поруч. Це пари документів, про які **точно**
відомо, що вони означають одне й те саме.

Беремо ті пари, у яких спільних слів майже немає (перетин за Жаккаром менший за
0.34) — рівно той відбір, який зробила [тема 05](../05-tfidf/lecture.html).

In [ ]:
NOISE = re.compile(r"[@<>]|https?://")

def paraphrase_pairs():
    """Пари українських перекладів того самого оригіналу, у яких майже немає спільних слів."""
    by_source = collections.defaultdict(set)
    for program, source, target in corpus:
        if source.strip() == 'translator-credits':
            continue
        by_source[source].add(' '.join(target.split()))
    found = []
    for source, variants in by_source.items():
        texts = sorted(variants)
        for i in range(len(texts)):
            for j in range(i + 1, len(texts)):
                first, second = texts[i], texts[j]
                if NOISE.search(first) or NOISE.search(second):
                    continue
                left, right = set(tokenize(first)), set(tokenize(second))
                if len(left) < 3 or len(right) < 3:
                    continue
                if len(left & right) / len(left | right) < 0.34:
                    found.append((first, second, len(left & right)))
    return found

pairs = paraphrase_pairs()
print(f'пар «те саме іншими словами»: {len(pairs)}')
print(f'з них без жодного спільного слова: {sum(1 for p in pairs if p[2] == 0)}')
print()
for first, second, shared in pairs[:3]:
    print(f'  спільних слів {shared}')
    print(f'    A: {first}')
    print(f'    B: {second}')

### Контроль, без якого замір нічого не вартий

Середній косинус синонімів сам по собі не означає нічого. Якщо новий спосіб підніме
синонімам косинус із 0.22 до 0.85, а заразом підніме й **випадковим** парам з 0.01
до 0.31 — він не почав бачити зміст, він просто зробив усе схожим на все.

Тому кожен замір цього зошита має дві колонки й **розрив** між ними. Контроль —
випадкові пари документів із вибірки на 20 000, і зерно міняє саме цю вибірку.

In [ ]:
SAMPLE_SIZE = 20000

def sample_documents(seed, n=SAMPLE_SIZE):
    """Ті самі документи при тому самому зерні — на будь-якій машині."""
    rng = np.random.default_rng(seed)
    picked = rng.choice(len(documents), min(n, len(documents)), replace=False)
    return [documents[i] for i in picked]

# контрольні пари: два незалежні випадкові набори документів тієї самої кількості
control_texts = {}
for seed in SEEDS:
    texts = sample_documents(seed)
    rng = np.random.default_rng(7 + seed)
    left = [texts[i] for i in rng.choice(len(texts), len(pairs))]
    right = [texts[i] for i in rng.choice(len(texts), len(pairs))]
    control_texts[seed] = (texts, left, right)

print(f'на кожне зерно: {len(control_texts[0][0])} документів вибірки, '
      f'{len(control_texts[0][1])} контрольних пар')
print('приклад контрольної пари при зерні 0:')
print('   A:', control_texts[0][1][0][:70])
print('   B:', control_texts[0][2][0][:70])

In [ ]:
tfidf_synonym, tfidf_random, tfidf_gap, tfidf_zeros = [], [], [], []

for seed in SEEDS:
    texts, left_texts, right_texts = control_texts[seed]
    vectorizer = TfidfVectorizer(token_pattern=TOKEN_PATTERN).fit(texts)
    left = vectorizer.transform([p[0] for p in pairs])
    right = vectorizer.transform([p[1] for p in pairs])
    same_meaning = np.asarray(left.multiply(right).sum(axis=1)).ravel()

    a = vectorizer.transform(left_texts)
    b = vectorizer.transform(right_texts)
    by_chance = np.asarray(a.multiply(b).sum(axis=1)).ravel()

    tfidf_synonym.append(same_meaning.mean())
    tfidf_random.append(by_chance.mean())
    tfidf_gap.append(same_meaning.mean() - by_chance.mean())
    tfidf_zeros.append(np.mean(same_meaning == 0))
    print(f'зерно {seed}: однаковий зміст {same_meaning.mean():.4f} · '
          f'випадкові {by_chance.mean():.4f} · розрив {tfidf_gap[-1]:.4f} · '
          f'рівно нуль у {100 * tfidf_zeros[-1]:.2f} %')

TFIDF_GAP = float(np.mean(tfidf_gap))
print()
print(f'TF-IDF по трьох зернах: однаковий зміст {np.mean(tfidf_synonym):.4f} · '
      f'випадкові {np.mean(tfidf_random):.4f}')
print(f'РОЗРИВ, який треба перемогти: {TFIDF_GAP:.4f} ±{np.std(tfidf_gap):.4f}')
print(f'рівно нуль дістає {100 * np.mean(tfidf_zeros):.2f} % пар однакового змісту')

## 6 · Матриця «слово × контекст» — спершу руками

Гіпотезу «слово пізнається за компанією» треба перетворити на числа. Робимо
найпряміше, що можна придумати: таблиця, де рядок — слово, колонка — теж слово, а в
клітинці стоїть, **скільки разів друге трапилось у вікні навколо першого**.

Шість рядків, вікно ±2. Перевірмо на них те, чого TF-IDF зробити не може за
побудовою: два слова, які **жодного разу не стояли поруч**, мають дістати
максимальну схожість.

In [ ]:
TOY = [
    'не вдалося відкрити файл',
    'не вдалося відкрити документ',
    'файл не знайдено',
    'документ не знайдено',
    'відкрити файл лише для читання',
    'відкрити документ лише для читання',
]
toy_docs = [line.split() for line in TOY]
toy_vocab = sorted({word for doc in toy_docs for word in doc})
toy_index = {word: i for i, word in enumerate(toy_vocab)}

def toy_counts(window):
    """Скільки разів колонка трапилась у вікні ±window навколо рядка."""
    size = len(toy_vocab)
    table = np.zeros((size, size))
    for doc in toy_docs:
        for position, word in enumerate(doc):
            low = max(0, position - window)
            high = min(len(doc) - 1, position + window)
            for other in range(low, high + 1):
                if other != position:
                    table[toy_index[word], toy_index[doc[other]]] += 1
    return table

toy = toy_counts(2)
for word in ('файл', 'документ'):
    row = toy[toy_index[word]]
    parts = [f'{toy_vocab[j]} {int(row[j])}' for j in np.argsort(-row) if row[j] > 0]
    print(f'{word:<9}: ' + ' · '.join(sorted(parts)))
print()
print('клітинка «файл — документ»:', int(toy[toy_index['файл'], toy_index['документ']]))

In [ ]:
def cosine_rows(table, first, second):
    """Косинус двох рядків таблиці: сума добутків, поділена на довжини."""
    a = table[toy_index[first]]
    b = table[toy_index[second]]
    if not a.any() or not b.any():
        return 0.0
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

def toy_ppmi(table):
    """Та сама таблиця, але в клітинці — PPMI замість кількості."""
    total = table.sum()
    row_sums = table.sum(axis=1, keepdims=True)
    col_sums = table.sum(axis=0, keepdims=True)
    expected = row_sums * col_sums / total
    with np.errstate(divide='ignore', invalid='ignore'):
        value = np.log(np.where(table > 0, table, 1) / np.where(expected > 0, expected, 1))
    return np.where(table > 0, np.maximum(value, 0.0), 0.0)

print('вікно | «файл · документ» | «не · для» сира | «не · для» PPMI')
for window in (1, 2, 3):
    counts = toy_counts(window)
    positive = toy_ppmi(counts)
    print(f'  ±{window}  |      {cosine_rows(counts, "файл", "документ"):.4f}       '
          f'|     {cosine_rows(counts, "не", "для"):.4f}      '
          f'|     {cosine_rows(positive, "не", "для"):.4f}')

assert abs(cosine_rows(toy_counts(2), 'файл', 'документ') - 1.0) < 1e-12, 'рядки мали збігтися!'
print()
print('✅ «файл» і «документ» дають рівно 1.0000 — а поруч не стояли жодного разу')

## 7 · Та сама матриця на всьому корпусі

Тепер по-справжньому: 6074 слова, вікно ±5. Робимо це не подвійним циклом, а
зсувом масиву — для кожної відстані `d` від 1 до 5 беремо всі пари «токен і токен
через `d` позицій» і лишаємо ті, що впали в один документ.

Матриця симетрична: кожну пару записуємо в обидві клітинки.

In [ ]:
# один довгий масив номерів слів плюс номер документа для кожної позиції
flat_words, flat_docs = [], []
for number, words in enumerate(kept):
    for word in words:
        flat_words.append(word_index[word])
        flat_docs.append(number)
flat_words = np.array(flat_words, dtype=np.int32)
flat_docs = np.array(flat_docs, dtype=np.int32)

def build_cooccurrence(window):
    """Матриця «слово × контекст»: у клітинці — скільки разів трапились поруч."""
    rows, cols = [], []
    for distance in range(1, window + 1):
        same_document = flat_docs[:-distance] == flat_docs[distance:]
        left = flat_words[:-distance][same_document]
        right = flat_words[distance:][same_document]
        rows += [left, right]          # пара рахується в обидва боки
        cols += [right, left]
    rows = np.concatenate(rows)
    cols = np.concatenate(cols)
    matrix = sp.coo_matrix((np.ones(len(rows)), (rows, cols)),
                           shape=(len(vocabulary), len(vocabulary))).tocsr()
    matrix.sum_duplicates()
    return matrix

clock = time.process_time()
counts5 = build_cooccurrence(5)
build_seconds = time.process_time() - clock
size = len(vocabulary)

print(f'матриця:            {size} × {size}')
print(f'клітинок усього:    {size * size}')
print(f'пар (слово, контекст): {int(counts5.sum())}')
print(f'ненульових клітинок:   {counts5.nnz}')
print(f'заповнено:             {100 * counts5.nnz / (size * size):.4f} %')
print(f'побудова зайняла {build_seconds:.2f} с процесорного часу')

### Дрібниця, що міняє число: коли саме викидати рідкісні слова

Ми викинули рідкісні слова **до** вікна. Якщо викинути їх після — слова, між якими
стояло викинуте, перестають бути сусідами, і пар виходить менше. Різницю варто
знати, інакше твоє число не зійдеться з чужим.

In [ ]:
all_words, all_docs = [], []
for number, words in enumerate(tokenized):
    for word in words:
        all_words.append(word_index.get(word, -1))     # -1 — слово поза словником
        all_docs.append(number)
all_words = np.array(all_words, dtype=np.int32)
all_docs = np.array(all_docs, dtype=np.int32)

pairs_after = 0
for distance in range(1, 6):
    good = ((all_docs[:-distance] == all_docs[distance:])
            & (all_words[:-distance] >= 0) & (all_words[distance:] >= 0))
    pairs_after += 2 * int(good.sum())

pairs_before = int(counts5.sum())
print(f'викидаємо ДО вікна:   {pairs_before} пар')
print(f'викидаємо ПІСЛЯ вікна: {pairs_after} пар')
print(f'різниця:               {pairs_before - pairs_after} пар')

## 8 · Чому сира частота не годиться

Матрицю збудовано — здавалося б, бери рядки й міряй косинус. Спробуймо. Ось десять
контекстів, які найчастіше стоять біля слова «файл».

In [ ]:
def top_contexts(matrix, word, how_many=10):
    """Найбільші клітинки рядка: (контекст, значення)."""
    row = matrix.getrow(word_index[word])
    order = np.argsort(-row.data)[:how_many]
    return [(vocabulary[row.indices[j]], row.data[j]) for j in order]

print('топ-10 контекстів слова «файл» за сирою частотою:')
print('  ' + ' · '.join(f'{context} {int(value)}' for context, value in top_contexts(counts5, 'файл')))

Шість із цих десяти — службові слова: вони стоять біля **всього** й нічого не кажуть
про «файл». Перевірмо, чи це особливість одного слова, чи властивість усієї матриці:
візьмімо топ-10 контекстів **кожного** з 6074 слів і порахуймо, яка частка позицій
дісталась службовим.

Список службових слів ми не завантажуємо з інтернету — він короткий і написаний
прямо тут, щоб було видно, що саме ми вважаємо службовим.

In [ ]:
FUNCTION_WORDS = {
    'не', 'для', 'у', 'з', 'і', 'та', 'в', 'на', 'що', 'це', 'як', 'до', 'за', 'від',
    'є', 'або', 'якщо', 'про', 'по', 'при', 'але', 'а', 'й', 'його', 'її', 'їх', 'ви',
    'ми', 'ти', 'я', 'він', 'вона', 'воно', 'цей', 'ця', 'ці', 'той', 'те', 'ті',
    'бути', 'має', 'можна', 'може',
}

def function_word_share(matrix, how_many=10):
    """Частка службових слів у топ-10 контекстів, усереднена по всьому словнику.

    Знаменник — рівно десять позицій на слово. У 54 слів контекстів менше десяти;
    порожні позиції ми не заповнюємо нічим, тож вони працюють на користь методу.
    """
    hits = 0
    for i in range(len(vocabulary)):
        row = matrix.getrow(i)
        if row.nnz == 0:
            continue
        order = np.argsort(-row.data)[:how_many]
        hits += sum(1 for j in order if vocabulary[row.indices[j]] in FUNCTION_WORDS)
    return 100 * hits / (how_many * len(vocabulary))

clock = time.process_time()
raw_function_share = function_word_share(counts5)
print(f'службових слів у топ-10 контекстів (сира частота): {raw_function_share:.2f} %')
print(f'заміряно за {time.process_time() - clock:.1f} с процесорного часу')

Майже половина того, що матриця «знає» про кожне слово, — це знання про те, що текст
українською. Наслідок вимірний: візьмімо сирі рядки як подання слів і повторімо
замір із розділу 5.

Документ подаємо середнім із векторів його слів. Одна подробиця важлива: **кожен
рядок слова спершу зводимо до одиничної довжини**, інакше частотні слова з довгими
рядками задавлять решту документа самою лише величиною.

In [ ]:
def document_vectors(word_vectors, texts):
    """Вектор документа — середнє з одиничних векторів його слів."""
    unit = normalize(word_vectors)             # кожне слово важить однаково
    out = np.zeros((len(texts), unit.shape[1]))
    for i, text in enumerate(texts):
        rows = [word_index[word] for word in tokenize(text) if word in word_index]
        if rows:
            block = unit[rows]
            out[i] = np.asarray(block.mean(axis=0)).ravel()
    lengths = np.linalg.norm(out, axis=1, keepdims=True)
    lengths[lengths == 0] = 1
    return out / lengths

def measure_gap(word_vectors, per_seed_vectors=None):
    """Середній косинус пар однакового змісту, випадкових пар і розрив між ними."""
    same, chance = [], []
    for seed in SEEDS:
        vectors = word_vectors if per_seed_vectors is None else per_seed_vectors[seed]
        left = document_vectors(vectors, [p[0] for p in pairs])
        right = document_vectors(vectors, [p[1] for p in pairs])
        same.append(float((left * right).sum(axis=1).mean()))
        texts, control_left, control_right = control_texts[seed]
        a = document_vectors(vectors, control_left)
        b = document_vectors(vectors, control_right)
        chance.append(float((a * b).sum(axis=1).mean()))
    gaps = [s - c for s, c in zip(same, chance)]
    return float(np.mean(same)), float(np.mean(chance)), float(np.mean(gaps)), float(np.std(gaps))

raw_result = measure_gap(counts5)
print(f'сира частота: однаковий зміст {raw_result[0]:.4f} · випадкові {raw_result[1]:.4f}')
print(f'              РОЗРИВ {raw_result[2]:.4f} ±{raw_result[3]:.4f}')
print(f'для порівняння TF-IDF: розрив {TFIDF_GAP:.4f}')
print()
print('косинус синонімів високий — і не означає нічого, бо випадкові пари поруч')

## 9 · PMI: ділимо на очікуване

Слово «не» трапилось біля «файл» 1371 раз — це багато чи мало? Без точки відліку
відповіді немає. Точку відліку дає проста думка: **скільки б їх стояло поруч, якби
вони не мали одне до одного стосунку?**

Якщо «не» становить 4 % усіх контекстних позицій, то й серед сусідств слова «файл»
ми очікували б побачити «не» приблизно в 4 % випадків. Ділимо фактичне на очікуване,
беремо логарифм — це і є **PMI**, поточкова взаємна інформація.

Порахуймо руками на двох прикладах, які дадуть протилежні відповіді.

In [ ]:
row_totals = np.asarray(counts5.sum(axis=1)).ravel()      # скільки сусідств у слова
col_totals = np.asarray(counts5.sum(axis=0)).ravel()      # скільки разів слово було контекстом
all_pairs = row_totals.sum()

def pmi_by_hand(word, context):
    """PMI однієї клітинки, порахований по кроках — саме так, як у лекції."""
    seen = counts5[word_index[word], word_index[context]]
    context_share = col_totals[word_index[context]] / all_pairs
    neighbours = row_totals[word_index[word]]
    expected = neighbours * context_share
    return seen, context_share, neighbours, expected, np.log(seen / expected)

seen, share, neighbours, expected, value = pmi_by_hand('файл', 'не')
print('приклад 1 · «файл» і найчастіший сусід «не»')
print(f'   «не» займає {100 * share:.2f} % усіх контекстних позицій')
print(f'   у «файла» {int(neighbours)} сусідств, отже очікувано {expected:.1f} раза')
print(f'   трапилось {int(seen)}, відношення {seen / expected:.2f}, PMI = {value:.4f}')

seen, share, neighbours, expected, value = pmi_by_hand('вікно', 'діалогове')
print()
print('приклад 2 · «вікно» і рідкісний сусід «діалогове»')
print(f'   у «вікна» {int(neighbours)} сусідств, очікувано {expected:.3f} раза')
print(f'   трапилось {int(seen)}, відношення {seen / expected:.0f}, PMI = {value:.4f}')
print('   за сирою частотою «діалогове» стояло аж четвертим:',
      ' · '.join(f'{c} {int(v)}' for c, v in top_contexts(counts5, 'вікно', 4)))

### PPMI: чому відрізаємо відʼємне

Відʼємний PMI означає «ці двоє трапляються поруч рідше, ніж випадково». Звучить як
корисна інформація — і в теорії так і є. На практиці її викидають, бо надійно
заміряти її нема з чого: у нас 3.6 мільйона пар на 36.9 мільйона клітинок, і
переважна більшість пар не трапилась разом жодного разу не тому, що уникає одна
одної, а тому, що тексту замало.

Обрізане знизу нулем називається **PPMI**.

In [ ]:
def ppmi_matrix(counts, shift=0.0):
    """PPMI: max(0, log(факт / очікуване) − зсув). Рахуємо лише в ненульових клітинках."""
    cells = counts.tocoo()
    rows_sum = np.asarray(counts.sum(axis=1)).ravel()
    cols_sum = np.asarray(counts.sum(axis=0)).ravel()
    total = rows_sum.sum()
    expected = rows_sum[cells.row] * cols_sum[cells.col] / total
    value = np.log(cells.data / expected) - shift
    keep = value > 0
    return sp.coo_matrix((value[keep], (cells.row[keep], cells.col[keep])),
                         shape=counts.shape).tocsr()

positive5 = ppmi_matrix(counts5)
size = len(vocabulary)
print(f'ненульових було:   {counts5.nnz} ({100 * counts5.nnz / (size * size):.4f} %)')
print(f'ненульових стало:  {positive5.nnz} ({100 * positive5.nnz / (size * size):.4f} %)')
print(f'відсікли клітинок: {counts5.nnz - positive5.nnz}')

Перевіримо, що наша формула — саме та, що написана в лекції, а не схожа на неї.
Порахуємо одну клітинку окремо, «в лоб», і порівняємо з тим, що лежить у матриці.

In [ ]:
by_hand = max(0.0, float(pmi_by_hand('вікно', 'діалогове')[4]))
from_matrix = float(positive5[word_index['вікно'], word_index['діалогове']])
assert np.isclose(by_hand, from_matrix), 'наша матриця розійшлася з ручним розрахунком!'
print(f'руками {by_hand:.6f} · з матриці {from_matrix:.6f}')
print('✅ збігається')

Тепер найцікавіше: та сама перевірка на службові слова. Одна формула, жодного списку
стоп-слів — і подивимось, що станеться з тими 41 %.

In [ ]:
positive_function_share = function_word_share(positive5)
print(f'службових у топ-10, сира частота: {raw_function_share:.2f} %')
print(f'службових у топ-10, PPMI:         {positive_function_share:.2f} %')
print()
print('топ-10 контекстів «файл» за PPMI:')
print('  ' + ' · '.join(f'{c} {v:.4f}' for c, v in top_contexts(positive5, 'файл')))

In [ ]:
ppmi_result = measure_gap(positive5)

print(f'{"подання":<28}{"синоніми":>10}{"випадкові":>12}{"розрив":>10}')
print(f'{"TF-IDF (тема 05)":<28}{np.mean(tfidf_synonym):>10.4f}{np.mean(tfidf_random):>12.4f}{TFIDF_GAP:>10.4f}')
print(f'{"сира частота, повний рядок":<28}{raw_result[0]:>10.4f}{raw_result[1]:>12.4f}{raw_result[2]:>10.4f}')
print(f'{"PPMI, повний рядок":<28}{ppmi_result[0]:>10.4f}{ppmi_result[1]:>12.4f}{ppmi_result[2]:>10.4f}')
print()
print(f'PPMI обійшла TF-IDF, але ледве: приріст {ppmi_result[2] - TFIDF_GAP:+.4f}')

## 10 · Гіпотеза Гарріса числом

Дотепер ми брали гіпотезу на віру. Перевірмо її прямо — але пари синонімів треба
звідкись узяти, і брати їх з голови не можна: підібраний список доводить лише смак
того, хто підбирав.

Здобуваємо їх із корпусу механічно. Беремо два переклади одного англійського рядка,
які відрізняються **рівно одним словом з кожного боку**, і оголошуємо ці два слова
парою однакового змісту. Викидаємо пари, що відрізняються лише апострофом або
збігаються першими чотирма літерами: це форми одного слова, а не синоніми.

In [ ]:
def without_apostrophes(word):
    for mark in ("'", 'ʼ', '’'):
        word = word.replace(mark, '')
    return word

def synonym_pairs():
    """Пари слів, які стоять на тому самому місці у двох перекладах одного рядка."""
    by_source = collections.defaultdict(set)
    for program, source, target in corpus:
        if source.strip() == 'translator-credits':
            continue
        by_source[source].add(' '.join(target.split()))
    found = set()
    for source, variants in by_source.items():
        texts = sorted(variants)
        for i in range(len(texts)):
            for j in range(i + 1, len(texts)):
                left, right = tokenize(texts[i]), tokenize(texts[j])
                if len(left) < 2 or len(right) < 2:
                    continue                       # однослівні підписи нічого не доводять
                only_left = set(left) - set(right)
                only_right = set(right) - set(left)
                if len(only_left) != 1 or len(only_right) != 1:
                    continue
                first, second = only_left.pop(), only_right.pop()
                if first not in word_index or second not in word_index:
                    continue
                if without_apostrophes(first) == without_apostrophes(second):
                    continue                       # та сама форма, інший апостроф
                if first[:4] == second[:4]:
                    continue                       # форми одного слова
                found.add(tuple(sorted((first, second))))
    return sorted(found)

synonyms = synonym_pairs()
print(f'пар однакового змісту, здобутих із корпусу: {len(synonyms)}')
for first, second in synonyms:
    print(f'   {first} · {second}')

Список не бездоганний: механічний спосіб пропустив кілька пар на кшталт
«запамʼятати · змінити», які синонімами не є. Але саме тому він і чесний — ми його
не чистили руками.

Тепер контроль. Порівнювати синоніми з **будь-якими** випадковими словами було б
нечесно: наші синоніми переважно частотні, а частотні слова мають щільніші рядки й
через це вищі косинуси самі по собі. Тому для кожної пари беремо випадкове слово
**такої самої частоти**, як друге слово пари: сусіда по частотному рангу.

In [ ]:
frequencies = np.array([form_counts[word] for word in vocabulary])
by_frequency = np.argsort(-frequencies)                  # номери слів від частих до рідкісних
rank_of = np.empty(len(vocabulary), dtype=int)
rank_of[by_frequency] = np.arange(len(vocabulary))
RANK_BAND = 10                                            # десять сусідів по рангу з кожного боку

def frequency_twin(word, avoid, rng):
    """Випадкове слово тієї самої частоти, що й задане."""
    rank = rank_of[word_index[word]]
    low = max(0, rank - RANK_BAND)
    high = min(len(vocabulary), rank + RANK_BAND + 1)
    choices = [by_frequency[k] for k in range(low, high)
               if vocabulary[by_frequency[k]] not in avoid]
    return vocabulary[choices[rng.integers(len(choices))]]

def word_cosine(matrix, first, second):
    unit = matrix                                        # матриця вже нормована рядками
    return float(unit[word_index[first]].multiply(unit[word_index[second]]).sum())

def harris_check(matrix, label):
    """Косинус пар однакового змісту проти контролю тієї самої частоти + AUC."""
    unit = normalize(matrix)
    same = [word_cosine(unit, a, b) for a, b in synonyms]
    controls, areas = [], []
    for seed in SEEDS:
        rng = np.random.default_rng(100 + seed)
        chance = [word_cosine(unit, a, frequency_twin(b, {a, b}, rng)) for a, b in synonyms]
        controls.append(float(np.mean(chance)))
        first = np.array(same)[:, None]
        second = np.array(chance)[None, :]
        areas.append(float(((first > second).sum() + 0.5 * (first == second).sum())
                           / (len(same) * len(chance))))
    print(f'{label:<14} синоніми {np.mean(same):.4f} · контроль {np.mean(controls):.4f} '
          f'±{np.std(controls):.4f} · AUC {np.mean(areas):.4f} ±{np.std(areas):.4f}')
    return float(np.mean(same)), float(np.mean(controls)), float(np.mean(areas)), float(np.std(areas))

harris_raw = harris_check(counts5, 'сира частота')
harris_ppmi = harris_check(positive5, 'PPMI')
print()
print(f'AUC відповідає на питання: якщо взяти навмання одну пару синонімів і одну')
print(f'контрольну, як часто синоніми виявляться схожішими. Вийшло '
      f'{100 * harris_ppmi[2]:.1f} % — це і є гіпотеза Гарріса, перетворена на число.')

### Звідки береться косинус

Косинус двох слів — це сума добутків по всіх 6074 контекстах, і майже всі доданки
нульові: два слова мають спільними лише кілька контекстів із тисяч. Розкладімо один
косинус на доданки й подивимось, хто його справді дає.

In [ ]:
def cosine_parts(first, second, how_many=6):
    """Доданки косинуса: спільні контексти двох слів і внесок кожного."""
    unit = normalize(positive5)
    a = unit.getrow(word_index[first])
    b = unit.getrow(word_index[second])
    shared = set(a.indices) & set(b.indices)
    parts = [(vocabulary[k], float(a[0, k] * b[0, k])) for k in shared]
    parts.sort(key=lambda item: -item[1])
    return parts, sum(value for _, value in parts)

parts, total = cosine_parts('регулярний', 'формальний')
print(f'пара «регулярний · формальний»: косинус {total:.4f}, спільних контекстів {len(parts)}')
for context, value in parts[:6]:
    print(f'   {context:<16} {value:.4f}   це {100 * value / total:.1f} % косинуса')

## 11 · Вікно як ручка

Ширина вікна виглядає дрібним параметром, який беруть за замовчуванням і забувають.
Насправді вона міняє не якість результату, а **вид схожості**, який ти дістаєш.

Щоб це побачити, потрібні найближчі сусіди слова — а для них потрібне подання, у
якому косинус має сенс. Забігаємо трохи вперед і беремо стиснене подання з
розділу 13: PPMI, стиснена сингулярним розкладом до 64 вимірів. Далі в темі ми його
збудуємо докладно, тут воно нам просто інструмент.

In [ ]:
def word_vectors_svd(matrix, dimensions, seed):
    """Три множники розкладу PPMI-матриці: U, Σ і Vᵀ. Подання слова — рядок U."""
    return randomized_svd(matrix, n_components=dimensions, random_state=seed)

def nearest(vectors, word, how_many=6):
    """Найближчі сусіди за косинусом у щільному поданні."""
    unit = normalize(vectors)
    scores = unit @ unit[word_index[word]]
    scores[word_index[word]] = -2                     # саме слово сусідом собі не є
    order = np.argsort(-scores)[:how_many]
    return [(vocabulary[i], float(scores[i])) for i in order]

windows = (2, 5, 10)
window_vectors = {}
window_counts = {}
clock = time.process_time()
for width in windows:
    window_counts[width] = build_cooccurrence(width)
    window_vectors[width] = word_vectors_svd(ppmi_matrix(window_counts[width]), 64, 0)[0]
print(f'три матриці й три розклади: {time.process_time() - clock:.1f} с процесорного часу')
print()
for word in ('файл', 'зберегти'):
    for width in (2, 10):
        names = ' · '.join(name for name, score in nearest(window_vectors[width], word))
        print(f'{word:<9} ±{width:<3}: {names}')
    print()

Різниця видно на око: при вузькому вікні сусіди «файла» — це те, чим його **можна
замінити** в реченні; при широкому — те, що з ним **роблять**. Перше — схожість за
роллю в реченні, друге — за темою.

Але «видно на око» не аргумент. Заміряймо двома числами:

* яка частка сусідів має **ту саму частину мови**, що й слово (це синтаксис) —
  частини мови розбирає `pymorphy3` із [теми 03](../03-morphology/lecture.html);
* яка частка сусідів **справді трапляється з ним в одному документі** (це тема).

In [ ]:
import pymorphy3
morph = pymorphy3.MorphAnalyzer(lang='uk')
parts_of_speech = np.array([(morph.parse(word)[0].tag.POS or '?') for word in vocabulary])

# матриця «слово × слово»: чи трапились колись в одному документі
present = sp.coo_matrix((np.ones(len(flat_words)), (flat_docs, flat_words)),
                        shape=(len(kept), len(vocabulary))).tocsr()
present.data[:] = 1
together = (present.T @ present).tocsr()
together.data[:] = 1
print(f'пар слів, що бували в одному документі: {together.nnz}')

def top_neighbours(vectors, how_many=10):
    """Номери десяти найближчих сусідів для кожного слова словника."""
    unit = normalize(vectors)
    size = len(vocabulary)
    result = np.zeros((size, how_many), dtype=np.int32)
    for start in range(0, size, 1000):
        block = unit[start:start + 1000] @ unit.T
        for row in range(block.shape[0]):
            block[row, start + row] = -2
        rough = np.argpartition(-block, how_many, axis=1)[:, :how_many]
        rows = np.arange(block.shape[0])[:, None]
        order = np.argsort(-block[rows, rough], axis=1)
        result[start:start + block.shape[0]] = rough[rows, order]
    return result

clock = time.process_time()
neighbour_lists = {width: top_neighbours(window_vectors[width]) for width in windows}
print(f'сусіди для всіх {len(vocabulary)} слів у трьох вікнах: '
      f'{time.process_time() - clock:.1f} с процесорного часу')

In [ ]:
print(f'{"вікно":<8}{"та сама частина мови":>24}{"в одному документі":>22}')
for width in windows:
    top = neighbour_lists[width]
    same_part = 100 * float(np.mean(parts_of_speech[top] == parts_of_speech[:, None]))
    rows = np.repeat(np.arange(len(vocabulary)), top.shape[1])
    same_document = 100 * float(np.mean(np.asarray(together[rows, top.ravel()]).ravel() > 0))
    print(f'±{width:<7}{same_part:>23.2f} %{same_document:>20.2f} %')
print()
print('вузьке вікно ставить поруч слова однієї частини мови — це синтаксис;')
print('широке ставить поруч слова з одного повідомлення — це тема')

In [ ]:
print('скільки з десяти сусідів збігається між вікнами:')
for first, second in ((2, 5), (2, 10), (5, 10)):
    left, right = neighbour_lists[first], neighbour_lists[second]
    shared = np.mean([len(set(left[i]) & set(right[i])) / left.shape[1]
                      for i in range(len(vocabulary))])
    print(f'   ±{first} проти ±{second}: {shared:.4f}')

Списки різні — але чи стає від цього краще або гірше? Заміряймо розрив на тих самих
209 парах для кожного вікна. Три зерна, бо тут є випадковість SVD.

In [ ]:
print(f'{"вікно":<8}{"синоніми":>10}{"випадкові":>12}{"розрив":>10}{"розкид":>10}')
window_gaps = {}
for width in windows:
    matrix = ppmi_matrix(window_counts[width])
    per_seed = {seed: word_vectors_svd(matrix, 64, seed)[0] for seed in SEEDS}
    same, chance, gap, spread = measure_gap(None, per_seed_vectors=per_seed)
    window_gaps[width] = (gap, spread)
    print(f'±{width:<7}{same:>10.4f}{chance:>12.4f}{gap:>10.4f}{spread:>10.4f}')
print()
spread_max = max(spread for gap, spread in window_gaps.values())
close = abs(window_gaps[2][0] - window_gaps[5][0])
print(f'±2 і ±5 нерозрізненні: різниця {close:.4f} при розкиді до {spread_max:.4f}')
print(f'±10 просідає на {window_gaps[5][0] - window_gaps[10][0]:.4f} — трохи більше за розкид,')
print('але поруч із приростом самого методу (більш ніж 0.2) це дрібниця.')
print('Вікно міняє КОГО метод вважає схожим, а не НАСКІЛЬКИ добре він це робить.')

## 12 · Короткі документи: вікно, яке не заповнюється

У нашого корпусу є властивість, яку не можна оминути: медіана — шість слів. Тож
запитаймо прямо: скільки позицій узагалі мають **повне** вікно, тобто по `W` слів з
обох боків, не впираючись у край документа?

In [ ]:
def window_fill(width):
    """Скільки позицій мають повне вікно і скільки сусідів має середня позиція."""
    full = positions = pairs_total = 0
    for words in kept:
        length = len(words)
        positions += length
        for i in range(length):
            low = max(0, i - width)
            high = min(length - 1, i + width)
            pairs_total += high - low
            if i - width >= 0 and i + width <= length - 1:
                full += 1
    return full, positions, pairs_total

print(f'{"вікно":<8}{"позицій із повним вікном":>26}{"сусідів на позицію":>22}{"з максимуму":>14}')
for width in (1, 2, 5, 10):
    full, positions, pairs_total = window_fill(width)
    print(f'±{width:<7}{100 * full / positions:>25.2f} %{pairs_total / positions:>22.4f}'
          f'{2 * width:>14}')

При вікні ±5 повне вікно має лише кожна восьма позиція. Розширювати вікно на такому
корпусі **майже безкоштовно й майже безкорисно**: пари ростуть повільно, бо їх нема
звідки взяти. Межа вікна тут — не параметр, а край документа.

Напрошується склеїти всі рядки однієї програми в один довгий текст. Робити цього не
можна: сусідні рядки у файлі перекладу — різні повідомлення різних частин програми.
Склеївши їх, ми **вигадаємо сусідство, якого в мові немає**.

## 13 · Скільки це важить у памʼяті

Матриця розріджена — 2.88 % заповнення. Порахуймо чесно, складанням трьох масивів
формату CSR, а не на око.

In [ ]:
size = len(vocabulary)
dense_bytes = size * size * 8                               # float64 у кожній клітинці
csr_bytes = (counts5.nnz * 8                                # data, float64
             + counts5.nnz * 4                              # indices, int32
             + (size + 1) * 4)                              # indptr, int32
print(f'клітинок:                    {size * size}')
print(f'заповнено:                   {100 * counts5.nnz / (size * size):.4f} %')
print(f'щільно, float64:             {dense_bytes / 1e6:.2f} МБ')
print(f'розріджено, CSR:             {csr_bytes / 1e6:.2f} МБ')
print(f'відношення:                  {dense_bytes / csr_bytes:.1f}×')
print(f'щільні вектори по 64 виміри: {size * 64 * 8 / 1e6:.2f} МБ')

## 14 · SVD: 6074 колонки в 64 числа

PPMI-рядок слова «файл» — це 6074 числа. Він точний, але поганий як подання: два
синоніми, що вживаються в схожих, але не тих самих контекстах, дістануть майже
нульовий косинус просто тому, що їхні ненульові клітинки не збіглися.

**Сингулярний розклад** (SVD) шукає 64 нові «запитання про слово», кожне з яких —
суміш старих колонок, і ці 64 відповіді описують слово майже так само добре, як усі
6074. Записується це так: M ≈ U · Σ · Vᵀ, де U — таблиця «слово × нове запитання»
(саме її ми заберемо), Σ — важливість кожного запитання, Vᵀ — як нові запитання
складено зі старих.

In [ ]:
clock = time.process_time()
left, sigma, right = word_vectors_svd(positive5, 256, 0)
svd_seconds = time.process_time() - clock

print(f'розклад на 256 вимірів: {svd_seconds:.1f} с процесорного часу')
print('перші пʼять чисел Σ:', ', '.join(f'{value:.2f}' for value in sigma[:5]))
print()
print(f'перше число більше за друге у {sigma[0] / sigma[1]:.2f} раза — це «запитання»,')
print('яке відповідає загальній частотності слова, і далі воно ще дасть про себе знати')

Перевіримо, що всередині немає магії — двома незалежними звірками з бібліотекою.

Перша: `randomized_svd` — метод **наближений**, він не рахує розклад чесно, а
вгадує його випадковими проєкціями. Візьмімо кут матриці 600 × 600, порахуймо
розклад **точно** через `numpy.linalg.svd` і порівняймо числа Σ.

Друга: наш косинус — це просто скалярний добуток нормованих рядків. Звірмо його з
`cosine_similarity` зі `scikit-learn`.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

corner = positive5[:600, :600]
exact = np.linalg.svd(corner.toarray(), compute_uv=False)[:16]
approximate = randomized_svd(corner, n_components=16, random_state=0)[1]
worst = float(np.max(np.abs(approximate - exact) / exact))

print('точні числа Σ:     ', ', '.join(f'{value:.4f}' for value in exact[:5]))
print('наші, наближені:   ', ', '.join(f'{value:.4f}' for value in approximate[:5]))
print(f'найгірше відхилення серед 16 чисел: {100 * worst:.2f} %')
assert np.allclose(approximate[:5], exact[:5], rtol=1e-3), 'розклад розійшовся з точним!'

ours = float(normalize(positive5)[word_index['файл']]
             .multiply(normalize(positive5)[word_index['документ']]).sum())
library = float(cosine_similarity(positive5[word_index['файл']],
                                  positive5[word_index['документ']])[0, 0])
assert np.isclose(ours, library), 'наш косинус розійшовся з бібліотечним!'
print(f'косинус «файл · документ»: наш {ours:.6f}, бібліотечний {library:.6f}')
print('✅ обидві звірки пройшли')

### Подробиця, що коштує половини результату

Природно взяти за вектор слова добуток **U · Σ** — так робить `fit_transform` і так
написано в більшості прикладів. Заміряймо всі три варіанти на нашій задачі.

In [ ]:
scalings = {'U (без ваг)': lambda u, s: u,
            'U · Σ^0.5': lambda u, s: u * np.sqrt(s),
            'U · Σ': lambda u, s: u * s}

decomposed = {seed: word_vectors_svd(positive5, 64, seed) for seed in SEEDS}
print(f'{"вектор слова":<16}{"синоніми":>10}{"випадкові":>12}{"розрив":>10}{"розкид":>10}')
for name, make in scalings.items():
    per_seed = {seed: make(u, s) for seed, (u, s, v) in decomposed.items()}
    same, chance, gap, spread = measure_gap(None, per_seed_vectors=per_seed)
    print(f'{name:<16}{same:>10.4f}{chance:>12.4f}{gap:>10.4f}{spread:>10.4f}')
print()
print('U · Σ має НАЙВИЩИЙ косинус синонімів — і найгірший розрив: множення на Σ')
print('підсилює перший, найчастотніший напрям, і всі вектори повертаються в один бік')

### Скільки вимірів брати

Тепер прожену сітку від 4 до 256 вимірів, по три зерна на точку. Це найдорожча
клітинка зошита — приблизно півхвилини процесорного часу.

In [ ]:
DIMENSIONS = (4, 8, 16, 32, 64, 128, 256)
curve = {}
clock = time.process_time()
for dimensions in DIMENSIONS:
    per_seed = {seed: word_vectors_svd(positive5, dimensions, seed)[0] for seed in SEEDS}
    curve[dimensions] = measure_gap(None, per_seed_vectors=per_seed)
sweep_seconds = time.process_time() - clock

print(f'{"вимірів":<10}{"синоніми":>10}{"випадкові":>12}{"розрив":>10}{"розкид":>10}')
for dimensions in DIMENSIONS:
    same, chance, gap, spread = curve[dimensions]
    mark = '  ←' if gap == max(v[2] for v in curve.values()) else ''
    print(f'{dimensions:<10}{same:>10.4f}{chance:>12.4f}{gap:>10.4f}{spread:>10.4f}{mark}')
print()
print(f'уся крива: {sweep_seconds:.1f} с процесорного часу')
best_gap = curve[32][2] - curve[64][2]
spreads = curve[32][3] + curve[64][3]
print(f'між 32 і 64 вимірами різниця {best_gap:.4f} при сумі розкидів {spreads:.4f}:')
print(f'   перевага 32 {"є" if best_gap > spreads else "губиться в розкиді"}, '
      f'але вона {curve[64][2] / best_gap:.0f} разів менша за сам приріст методу')
print('далі беремо 64, бо стільки ж вимірів має skip-gram теми 12')

Крива має максимум, і це не випадковість. Мало вимірів — модель не встигає описати
різницю між словами (при чотирьох вимірах вони описують хіба що частотність, і всі
пари злипаються). Багато — вона знову починає запамʼятовувати випадкові збіги
окремих клітинок. Це та сама рівновага, яку курс машинного навчання зве компромісом
між зміщенням і дисперсією.

Але вершина **пласка**, і сказати це чесніше, ніж називати переможця.

### Спільний напрямок, у який дивляться всі вектори

Чому ділення на Σ так допомагає, видно з одного числа: середнього косинуса між
**усіма** словами словника. Якщо він далеко від нуля, значить, у векторів є спільний
напрямок, у який вони дивляться разом. Цей напрямок додається до кожної пари —
і до синонімів, і до випадкових слів, — тож він не розрізняє нічого, а лише завищує
все.

In [ ]:
def average_cosine(vectors, block_size=1000):
    # середній косинус між усіма парами слів словника; блоками, щоб не тримати
    # в памʼяті квадратну матрицю на 36.9 мільйона чисел
    unit = normalize(vectors)
    size = unit.shape[0]
    total = 0.0
    for start in range(0, size, block_size):
        block = unit[start:start + block_size] @ unit.T
        for row in range(block.shape[0]):
            block[row, start + row] = 0.0            # слово із самим собою не рахуємо
        total += float(block.sum())
    return total / (size * (size - 1))

left64, sigma64, right64 = decomposed[0]
for name, vectors in (('U (без ваг)', left64), ('U · Σ', left64 * sigma64)):
    print(f'{name:<12} середній косинус між усіма словами словника: '
          f'{average_cosine(vectors):.4f}')
print()
print('U · Σ тягне всі слова в один бік — саме тому в нього і високий косинус')
print('синонімів, і такий самий високий у випадкових пар')

### Два боки розкладу: лівий множник і сума

У розкладі M ≈ U · Σ · Vᵀ два набори векторів: **U** — по одному на слово-ціль,
**V** — по одному на слово-контекст. Skip-gram [теми 12](../12-word2vec/lecture.html)
влаштований так само: у нього теж дві матриці, центральна й контекстна, і Word2Vec
за замовчуванням віддає лише центральну, хоча суму двох часто беруть як краще
подання.

Порівнювати треба однакове з однаковим, тож заміряймо обидва боки й у себе. Одна
чесна засторога наперед: наша матриця **симетрична** — ми записуємо кожну пару в
обидві клітинки, — тож U і V у нас майже той самий набір напрямів, і сума лише
гасить ті з них, що відповідають відʼємним власним числам.

In [ ]:
sides = {
    'лівий множник U': {seed: left for seed, (left, sigma, right) in decomposed.items()},
    'сума U + V': {seed: left + right.T for seed, (left, sigma, right) in decomposed.items()},
}
print(f'{"подання":<20}{"синоніми":>10}{"випадкові":>12}{"розрив":>10}{"розкид":>10}')
side_gaps = {}
for name, per_seed in sides.items():
    same, chance, gap, spread = measure_gap(None, per_seed_vectors=per_seed)
    side_gaps[name] = gap
    print(f'{name:<20}{same:>10.4f}{chance:>12.4f}{gap:>10.4f}{spread:>10.4f}')

difference = [float(np.abs(left64[:, k] - right64[k]).max()) for k in range(6)]
print()
print('найбільша різниця між колонкою U і рядком V, перші шість напрямів:')
print('   ' + '  '.join(f'{value:.4f}' for value in difference))
print('нулі означають, що напрям той самий: матриця симетрична, і «два боки»')
print('у нас куди менш самостійні, ніж дві матриці skip-gram')

## 15 · Замір: чи пробито стіну

Збираємо все в одну таблицю. Порівнювати можна тільки останню колонку: абсолютні
косинуси різних подань живуть у різних просторах із різною типовою відстанню.

In [ ]:
# Числа skip-gram цитовані з теми 12, а не заміряні тут. Тримаємо їх у змінних:
# коли тема 12 перезаміряє їх, правити доведеться рівно ці два рядки.
SKIPGRAM_CENTRAL = 0.3093        # тільки центральна матриця — те, що Word2Vec віддає
SKIPGRAM_SUM = 0.5390            # сума центральної й контекстної, v + u

final = [
    ('TF-IDF (тема 05)', np.mean(tfidf_synonym), np.mean(tfidf_random), TFIDF_GAP),
    ('сира частота, рядок', raw_result[0], raw_result[1], raw_result[2]),
    ('PPMI, повний рядок', ppmi_result[0], ppmi_result[1], ppmi_result[2]),
    ('PPMI + SVD 32, U', curve[32][0], curve[32][1], curve[32][2]),
    ('PPMI + SVD 64, U', curve[64][0], curve[64][1], curve[64][2]),
]
print(f'{"подання":<26}{"синоніми":>10}{"випадкові":>12}{"розрив":>10}')
for name, same, chance, gap in final:
    print(f'{name:<26}{same:>10.4f}{chance:>12.4f}{gap:>10.4f}')
print(f'{"PPMI + SVD 64, U + V":<26}{"":>10}{"":>12}{side_gaps["сума U + V"]:>10.4f}')
print(f'{"skip-gram 64, центральна":<26}{0.7054:>10.4f}{0.3961:>12.4f}{SKIPGRAM_CENTRAL:>10.4f}')
print(f'{"skip-gram 64, v + u":<26}{0.5628:>10.4f}{0.0238:>12.4f}{SKIPGRAM_SUM:>10.4f}')

ours_left = curve[64][2]
ours_sum = side_gaps['сума U + V']
print()
print(f'розрив виріс із {TFIDF_GAP:.4f} до {ours_left:.4f} — у {ours_left / TFIDF_GAP:.2f} раза,')
print(f'і розкид по зернах ({curve[64][3]:.4f}) на порядок менший за приріст')
print()
print('ПОРІВНЯННЯ З НАВЧЕНИМИ ЕМБЕДИНГАМИ · порівнюємо однакове з однаковим:')
print(f'   один бік розкладу проти центральної матриці: '
      f'{ours_left:.4f} проти {SKIPGRAM_CENTRAL:.4f} — це '
      f'{100 * ours_left / SKIPGRAM_CENTRAL:.0f} %, тобто підрахунок ПОПЕРЕДУ')
print(f'   сума двох боків проти v + u:                 '
      f'{ours_sum:.4f} проти {SKIPGRAM_SUM:.4f} — це '
      f'{100 * ours_sum / SKIPGRAM_SUM:.0f} %, тут навчання попереду на '
      f'{SKIPGRAM_SUM - ours_sum:.4f}')
print()
print('порівнювати лівий множник із сумою двох матриць було б підтасовуванням')
print('у будь-який бік — тому таблиця має шість рядків, а не чотири')

### Зсунута PPMI: механізм теми 12, який тут шкодить

У літературі показано, що skip-gram із негативним семплюванням на `k` прикладів
неявно розкладає **зсунуту** PPMI-матрицю: max(0, PMI − log k). Ми вміємо це
порахувати — у нашій `ppmi_matrix` для цього є параметр `shift`.

Це число лекція називає у врізці розділу «PMI», але порахувати його можна лише
тепер, коли SVD вже написаний.

In [ ]:
print(f'{"зсув":<12}{"ненульових":>12}{"розрив":>10}{"розкид":>10}')
for shift, name in ((0.0, 'без зсуву'), (np.log(2), 'log 2'), (np.log(5), 'log 5')):
    shifted = ppmi_matrix(counts5, shift=shift)
    per_seed = {seed: word_vectors_svd(shifted, 64, seed)[0] for seed in SEEDS}
    same, chance, gap, spread = measure_gap(None, per_seed_vectors=per_seed)
    print(f'{name:<12}{shifted.nnz:>12}{gap:>10.4f}{spread:>10.4f}')
print()
print('на нашому корпусі зсув ШКОДИТЬ: те, що тема 12 матиме за замовчуванням,')
print('тут працює проти нас — і це перше, чого не варто чекати від наступної теми')

## 16 · Межа підходу: антоніми

Тепер найважливіше застереження теми, і воно стосується не нашої реалізації, а самої
гіпотези Гарріса. «Увімкнути» і «вимкнути» стоять у тих самих реченнях, біля тих
самих слів, на тому самому місці: «увімкнути звук» і «вимкнути звук». Оточення
схоплює **роль слова**, а не його знак.

In [ ]:
ANTONYMS = [('вгору', 'вниз'), ('увімкнути', 'вимкнути'), ('відкрити', 'закрити'),
            ('додати', 'вилучити'), ('увімкнено', 'вимкнено'), ('більше', 'менше'),
            ('показати', 'сховати'), ('успішно', 'помилково')]
unit_ppmi = normalize(positive5)

antonym_scores = [(a, b, word_cosine(unit_ppmi, a, b)) for a, b in ANTONYMS]
for a, b, value in sorted(antonym_scores, key=lambda item: -item[2]):
    print(f'   {a + " · " + b:<26}{value:.4f}')

print()
print('для порівняння — дві пари синонімів із того самого списку:')
for a, b in (('та', 'і'), ('регулярного', 'формального')):
    print(f'   {a + " · " + b:<26}{word_cosine(unit_ppmi, a, b):.4f}')

rng = np.random.default_rng(100)
antonym_control = float(np.mean([word_cosine(unit_ppmi, a, frequency_twin(b, {a, b}, rng))
                                 for a, b in ANTONYMS]))
print()
print(f'середнє по восьми парах антонімів: {np.mean([v for _, _, v in antonym_scores]):.4f}')
print(f'середнє по 38 парах синонімів:     {harris_ppmi[0]:.4f}')
print(f'контроль тієї самої частоти:       {antonym_control:.4f}')
print()
print('АНТОНІМИ ВИЯВЛЯЮТЬСЯ СХОЖІШИМИ ЗА СИНОНІМИ — і це не дефект коду,')
print('а прямий наслідок того, що ми міряємо')

Практичний наслідок різкий: **ембединги не можна питати про заперечення**. Пошук за
схожістю на запит «увімкнути автозбереження» знайде інструкцію «як вимкнути
автозбереження» й покаже її першою. Жодне збільшення корпусу цього не виправить —
це властивість підходу, а не браку даних, і [тема 12](../12-word2vec/lecture.html)
успадкує її повністю.

Для порівняння покажімо ту саму пару в стисненому поданні — чи рятує SVD.

In [ ]:
sixty_four = normalize(decomposed[0][0])
print(f'{"пара":<26}{"PPMI":>10}{"PPMI + SVD 64":>16}')
for a, b in (('увімкнути', 'вимкнути'), ('регулярний', 'формальний'), ('файл', 'пароль')):
    dense_value = float(sixty_four[word_index[a]] @ sixty_four[word_index[b]])
    print(f'{a + " · " + b:<26}{word_cosine(unit_ppmi, a, b):>10.4f}{dense_value:>16.4f}')
print()
print('стиснення нічого тут не лікує: воно передає далі те саме, що знайшло')

## 17 · Що ми зробили

Порахували, хто з ким стоїть поруч. Поділили на те, скільки їх стояло б випадково.
Стиснули. Три дії — і подання, яке бачить зміст там, де десять тем поспіль ми бачили
лише збіг написань.

**Нічого не навчалось.** Один бік нашого розкладу випереджає центральну матрицю
навченого skip-gram, а сума двох боків поступається його сумі — але поступається,
а не програє вщент. Різниця між підрахунком і навчанням на цьому корпусі менша, ніж
про неї заведено говорити.

Те, чого підрахунок не вміє, ми теж заміряли й назвали: антоніми. І вони нікуди не
подінуться в наступній темі, бо вона вчиться на тій самій гіпотезі.

In [ ]:
print(f'усього зошит витратив {time.process_time() - started_at:.0f} с процесорного часу')
print()
print('ГОЛОВНІ ЧИСЛА ТЕМИ')
print(f'   документів               {len(documents)}')
print(f'   словник min_count=10     {len(vocabulary)} слів, покриття '
      f'{100 * covered / occurrences:.2f} %')
print(f'   пар (слово, контекст)    {int(counts5.sum())} при вікні ±5')
print(f'   розрив TF-IDF            {TFIDF_GAP:.4f}')
print(f'   розрив PPMI              {ppmi_result[2]:.4f}')
print(f'   розрив PPMI + SVD 64     {curve[64][2]:.4f} (U) · '
      f'{side_gaps["сума U + V"]:.4f} (U + V)')
print(f'   AUC гіпотези Гарріса     {harris_ppmi[2]:.4f}')
print(f'   антоніми проти синонімів '
      f'{np.mean([v for _, _, v in antonym_scores]):.4f} проти {harris_ppmi[0]:.4f}')

## 18 · Завдання

### 🟢 Рівень 1 — База

Візьми чотири слова, яких немає серед прикладів теми (наприклад «пароль», «диск»,
«шрифт», «принтер»), і надрукуй для кожного топ-10 контекстів **за сирою частотою**
і **за PPMI** поруч. Порахуй, скільки службових слів у кожному списку.

**Зроблено, якщо:** для кожного з чотирьох слів названо число службових слів в обох
списках і словами пояснено, чому воно так падає.

### 🟡 Рівень 2 — Плюс

Матриця «слово × контекст» у нас симетрична: ми записуємо пару в обидві клітинки.
Зроби **несиметричну** — окремо «контекст ліворуч» і «контекст праворуч», тобто
вектор довжиною 2 × 6074. Заміряй розрив на 209 парах, три зерна.

**Зроблено, якщо:** таблиця з двох рядків (симетрична / роздільна) із розривом і
розкидом, і висновок: чи вийшла різниця більшою за розкид.

### 🔴 Рівень 3 — Виклик

Наш контроль у розділі 10 бере слово тієї самої **частоти**. Це один спосіб бути
чесним, але не єдиний. Побудуй другий контроль: слово тієї самої **частини мови** й
приблизно тієї самої частоти — і подивись, чи впаде AUC гіпотези Гарріса.

**Зроблено, якщо:** AUC названо для обох контролів по трьох зернах, і сказано, який
з них суворіший і чому.

### Підказки

* Топ-10 контекстів уже вміє друкувати `top_contexts(matrix, word)` — їй байдуже,
  сира матриця чи PPMI.
* Для несиметричної матриці не треба нового коду вікна: збери дві матриці, `left`
  і `right`, і склей їх через `sp.hstack`.
* `parts_of_speech` уже порахований для всього словника — контроль за частиною мови
  будується з нього й `by_frequency` за пʼять рядків.